# NPO local third run — LoRA + quantization audit

Run the tested Python implementation beside this notebook with the **same kernel** as your existing FYP CUDA environment. Edit `config.json` first. The default runs three seeds, source-style sampled evaluation, legacy greedy answers, a longer generation audit, merged-checkpoint INT8/NF4, and both fixed/matched geometric Forget Quality.

This is a LoRA adaptation. Read `README.md`, the human audit rubric, and decision DL-024. Existing second-run evidence remains immutable. No actual third-run results are prefilled.

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

import pandas as pd
from IPython.display import HTML, display

HERE = Path.cwd()
if not (HERE / "run_experiment.py").is_file():
    HERE = HERE / "experiments/reproductions/NPO_3rd-test"
assert (HERE / "run_experiment.py").is_file(), (
    "Open this notebook from the third-run folder or repository root."
)
CONFIG = HERE / "config.json"
print("Kernel:", sys.executable)
display(pd.Series(json.loads(CONFIG.read_text())))

In [ ]:
# Meaningful CPU tests plus actual CUDA smoke tests when CUDA/bitsandbytes are available.
# A test failure raises before the real experiment starts.
subprocess.run(
    [sys.executable, "-m", "pytest", "tests", "-q", "--junitxml=validation.xml"],
    cwd=HERE,
    check=True,
)

In [ ]:
# Set only for an interrupted run with the same code, configuration and packages.
RESUME_RUN = None  # e.g. r"C:\FYP\runs\20261103_..."
command = [sys.executable, "-u", "run_experiment.py", "--config", str(CONFIG.resolve())]
if RESUME_RUN:
    command += ["--resume", str(RESUME_RUN)]
# Inherits stdout/stderr so training/evaluation progress is visible.
subprocess.run(command, cwd=HERE, check=True)

In [ ]:
settings = json.loads(CONFIG.read_text())
output_root = Path(settings["output_root"])
if not output_root.is_absolute():
    output_root = HERE / output_root
RUN_DIR = Path(RESUME_RUN) if RESUME_RUN else sorted(output_root.iterdir())[-1]
print("Results:", RUN_DIR)
display(pd.read_csv(RUN_DIR / "all_seeds_metrics.csv"))
display(pd.read_csv(RUN_DIR / "cross_seed_mean_std.csv", header=[0, 1]))
for seed in settings["seeds"]:
    folder = RUN_DIR / f"seed_{seed}"
    print("Seed", seed)
    display(pd.read_csv(folder / "epoch_metrics.csv"))
    display(pd.read_csv(folder / "forget_quality_fixed_and_matched.csv"))
    display(pd.read_csv(folder / "paired_directional_recovery.csv"))
    display(pd.read_csv(folder / "merge_drift.csv"))
    display(HTML((folder / "report.html").read_text(encoding="utf-8")))

In [ ]:
# Inspect complete model-generated answers from one seed, including every model condition.
seed_folder = RUN_DIR / f"seed_{settings['seeds'][0]}"
answers = pd.read_csv(seed_folder / "answers_side_by_side.csv")
with pd.option_context("display.max_colwidth", None):
    display(answers.query("split == 'forget05'").head(10))
print("Complete HTML:", seed_folder / "answers_side_by_side.html")
print("Human labels (blank until reviewed):", seed_folder / "human_audit_template.csv")